In [1]:
import datetime
import logging  # 追加：ログ制御用
import warnings  # 追加：警告非表示用
import pandas as pd
import yfinance as yf
from IPython.display import display, HTML

# 警告メッセージ（FutureWarningなど）を非表示にする
warnings.filterwarnings("ignore")

# yfinanceの内部エラー出力を非表示にする
yf_logger = logging.getLogger('yfinance')
yf_logger.setLevel(logging.CRITICAL)

# Define the lookback periods (in business days) for performance calculation.
LOOKBACK_DAYS = {
    "1 Day (%)": -2,
    "3 Days (%)": -4,
    "1 Week (%)": -6,
    "1 Month (%)": -21,
    "6 Months (%)": -126,
    "1 Year (%)": -252,
}

# =====================================================================
# DATA PROCESSING FUNCTION (Supports Market Segmentation)
# =====================================================================
def show_market_performances(market_dict):
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=60)
    
    # Format numerical values to 2 decimal places
    format_config = {"Latest Price": "{:,.2f}"}
    for column_name in LOOKBACK_DAYS.keys():
        format_config[column_name] = "{:+.2f}%"
        
    # ハイライト用の条件関数
    def highlight_extreme_values(val):
        if pd.isna(val):
            return ''
        if val >= 10:
            return 'background-color: #ffcccc; color: #cc0000; font-weight: bold;'
        elif val <= -10:
            return 'background-color: #cce5ff; color: #004085; font-weight: bold;'
        return ''
        
    # Process each market section
    for market_name, ticker_dict in market_dict.items():
        records = []
        
        # Display market header
        display(HTML(f"<h3 style='margin-top: 20px; color: #333;'>■ {market_name} Market</h3>"))
        
        for name, ticker in ticker_dict.items():
            try:
                # 【修正】 ignore_tz=True を追加してタイムゾーン警告を抑制
                df = yf.download(ticker, start=start_date, end=today, progress=False, ignore_tz=True)
                
                if df.empty:
                    # 元のコードにあったWarning表示も邪魔ならコメントアウトしてください
                    # print(f"[Warning] No data found for: {name} ({ticker})")
                    continue
                    
                prices = df["Close"].squeeze()
                latest_price = prices.iloc[-1]
                
                row = {
                    "Name": name,
                    "Ticker": ticker,
                    "Latest Price": latest_price
                }
                
                for label, index in LOOKBACK_DAYS.items():
                    if len(prices) >= abs(index):
                        past_price = prices.iloc[index]
                    else:
                        past_price = prices.iloc[0]
                    
                    row[label] = ((latest_price - past_price) / past_price) * 100
                    
                records.append(row)
                
            except Exception as e:
                # エラー時のprintも非表示にしたい場合はコメントアウトしてください
                pass
        
        if records:
            df_performance = pd.DataFrame(records)
            
            # .map を使用
            styled_df = (df_performance.style
                         .format(format_config)
                         .map(highlight_extreme_values, subset=list(LOOKBACK_DAYS.keys()))
                         .hide(axis="index"))
            
            display(styled_df)
        else:
            print("No data available for this market.")

In [2]:
# =====================================================================
# CONFIGURATION & EXECUTION (Segmented by Market)
# =====================================================================
MARKET_TICKERS = {
    "United States": {
        "S&P 500": "^GSPC",
        "NASDAQ Composite": "^IXIC",
        "Apple": "AAPL",
        "NVIDIA": "NVDA",
        "Microsoft": "MSFT",
    },
    "Japan": {
        "Nikkei 225": "^N225",
        "Toyota Motor": "7203.T",
        "Sony Group": "6758.T",
        "Mitsubishi UFJ Financial": "8306.T",
    },
    "Singapore": {
        "STI Index": "^STI",
        "DBS Group": "D05.SI",
        "UOB": "U11.SI",
        "Singtel": "Z74.SI",
        "CapitaLand Ascendas REIT": "A17U.SI",
    },
    "Japan Stock": {
        "小松製作所 (6301)": "6301.T",
        "ニデック (6594)": "6594.T",
        "未来工業 (7931)": "7931.T",
        "東部ネットワーク (9036)": "9036.T",
        "ニトリHD (9843)": "9843.T",
        "MRK HD (9980)": "9980.T",
    },
    "US Stock (ETF)": {
        "iシェアーズ コア米国総合債券ETF (AGG)": "AGG"
    }
}

# Run and display the segmented tables
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
S&P 500,^GSPC,"7,543.64",+0.81%,+0.08%,+0.81%,+2.13%,+1.76%,+1.76%
NASDAQ Composite,^IXIC,"26,206.89",+1.30%,+0.33%,+0.64%,+2.06%,-0.26%,-0.26%
Apple,AAPL,316.22,+0.90%,+1.14%,+7.42%,+8.83%,+8.04%,+8.04%
NVIDIA,NVDA,202.78,-0.66%,+3.70%,+2.63%,-2.60%,-7.59%,-7.59%
Microsoft,MSFT,384.36,+0.27%,-0.62%,+0.02%,-4.72%,-6.66%,-6.66%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Nikkei 225,^N225,"67,743.85",+1.38%,-2.86%,-1.44%,+5.49%,+8.53%,+8.53%
Toyota Motor,7203.T,"2,824.00",-2.25%,-3.39%,+1.11%,+2.78%,-1.60%,-1.60%
Sony Group,6758.T,"3,409.00",-1.70%,-0.38%,+2.37%,+1.19%,+1.10%,+1.10%
Mitsubishi UFJ Financial,8306.T,"3,414.00",-0.55%,+1.31%,+3.27%,+8.69%,+19.52%,+19.52%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
STI Index,^STI,"5,433.88",+1.20%,+3.31%,+4.15%,+8.94%,+9.94%,+9.94%
DBS Group,D05.SI,70.23,+1.64%,+4.96%,+5.72%,+12.19%,+19.50%,+19.50%
UOB,U11.SI,44.34,+2.33%,+9.43%,+10.66%,+16.50%,+19.87%,+19.87%
Singtel,Z74.SI,4.39,+0.46%,-1.35%,-1.57%,+3.29%,-7.58%,-7.58%
CapitaLand Ascendas REIT,A17U.SI,2.49,+0.00%,-0.40%,+0.40%,-0.80%,+0.40%,+0.40%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
小松製作所 (6301),6301.T,"6,395.00",-1.99%,-5.25%,+0.08%,-0.16%,-3.33%,-3.33%
ニデック (6594),6594.T,"2,613.00",+1.16%,-3.51%,-2.43%,+2.55%,-4.98%,-4.98%
未来工業 (7931),7931.T,"3,210.00",+0.47%,+0.94%,+2.72%,+6.64%,+2.72%,+2.72%
東部ネットワーク (9036),9036.T,"1,288.00",+1.42%,+0.78%,+2.38%,+6.01%,+4.04%,+4.04%
ニトリHD (9843),9843.T,"2,426.00",-0.68%,+2.15%,+3.08%,-8.99%,+2.71%,+2.71%
MRK HD (9980),9980.T,96.00,+0.00%,+1.05%,+3.23%,+0.00%,-3.03%,-3.03%


Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
iシェアーズ コア米国総合債券ETF (AGG),AGG,98.18,+0.14%,-0.49%,-0.32%,+0.14%,-0.42%,-0.42%


In [3]:
MARKET_TICKERS = {
    "Singapore Healthcare": {
        # --- 病院経営・総合クリニック ---
        "Raffles Medical Group (ラッフルズ医療)": "BSL.SI",  # 【修正】R01.SI から変更
        "IHH Healthcare (マウント・エリザベス等経営)": "Q0F.SI",  # 【修正】I01.SI から変更
        "Thomson Medical Group (トムソン医療)": "A50.SI",
        
        # --- 医療不動産（REIT / 病院・介護施設） ---
        "Parkway Life REIT (パークウェイ・ライフ)": "C2PU.SI",
        "First REIT (ファースト・リート)": "AW9U.SI",
        
        # --- 専門医療・歯科クリニックチェーン ---
        "Q&M Dental Group (歯科大手チェーン)": "QC7.SI",
        "Alliance Healthcare Group (総合診療・クリニック)": "MIJ.SI",  # 【修正】1D8から変更
        "Medinex Limited (医療サポート・クリニック運営)": "OTX.SI",
        
        # --- 医療用消耗品・製薬 ---
        "Medtecs International (医療用消耗品)": "546.SI",
        "Hyphens Pharma (医薬品・皮膚科製品)": "1J5.SI",
        "Riverstone": "AP4.SI"
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Raffles Medical Group (ラッフルズ医療),BSL.SI,0.94,+2.73%,+2.17%,+2.17%,+1.08%,-2.08%,-2.08%
IHH Healthcare (マウント・エリザベス等経営),Q0F.SI,2.78,-2.46%,-0.36%,+1.83%,+1.09%,-3.14%,-3.14%
Thomson Medical Group (トムソン医療),A50.SI,0.05,+0.00%,+0.00%,+0.00%,-1.85%,-7.02%,-7.02%
Parkway Life REIT (パークウェイ・ライフ),C2PU.SI,4.09,-0.49%,-0.73%,+0.25%,+3.28%,+2.51%,+2.51%
First REIT (ファースト・リート),AW9U.SI,0.23,+2.22%,+0.00%,+0.00%,+2.22%,-4.17%,-4.17%
Q&M Dental Group (歯科大手チェーン),QC7.SI,0.56,-0.89%,+1.83%,+0.91%,-6.72%,-5.93%,-5.93%
Alliance Healthcare Group (総合診療・クリニック),MIJ.SI,0.16,+0.00%,+0.00%,-3.07%,+0.00%,-3.07%,-3.07%
Medinex Limited (医療サポート・クリニック運営),OTX.SI,0.23,+0.00%,+0.00%,+4.55%,+9.52%,+0.00%,+0.00%
Medtecs International (医療用消耗品),546.SI,0.11,-0.88%,-0.88%,-1.74%,-6.61%,-19.86%,-19.86%
Hyphens Pharma (医薬品・皮膚科製品),1J5.SI,0.35,+0.00%,+0.00%,+1.45%,+7.69%,+4.48%,+4.48%


In [4]:
MARKET_TICKERS = {
"Singapore Food & Agribusiness": {
        "Wilmar International (ウィルマー・世界最大級のパーム油)": "F34.SI",
        "Olam Group (オラム・ココア、コーヒー、ナッツ等大手)": "VC2.SI",
        "Riverstone": "AP4.SI",
        "ヤンジジャン・シップビルディング(BS6)": "BS6.SI",
        "Sembcorp Ind (U96)": "U96.SI",
    }
}
show_market_performances(MARKET_TICKERS)

Name,Ticker,Latest Price,1 Day (%),3 Days (%),1 Week (%),1 Month (%),6 Months (%),1 Year (%)
Wilmar International (ウィルマー・世界最大級のパーム油),F34.SI,3.81,+0.53%,+2.42%,+2.97%,+7.32%,+4.10%,+4.10%
Olam Group (オラム・ココア、コーヒー、ナッツ等大手),VC2.SI,1.19,+0.00%,-3.25%,-1.65%,-8.46%,-2.46%,-2.46%
Riverstone,AP4.SI,0.85,+3.05%,+0.60%,+0.00%,-1.74%,+3.05%,+3.05%
ヤンジジャン・シップビルディング(BS6),BS6.SI,3.41,-0.58%,-4.21%,-1.16%,-1.73%,-12.79%,-12.79%
Sembcorp Ind (U96),U96.SI,5.60,-1.06%,-2.61%,-9.53%,-8.65%,-9.39%,-9.39%


<div style="background-color: #EBCB8B; color: #2E3440; padding: 12px; border-radius: 5px; font-weight: bold;">
    ⚠️ test test
</div>